In [1]:
import sys
import torch
import numpy as np
sys.path.append('../')
from GeCoSleep import *
from baselines import *

In [7]:
from tqdm import tqdm
from sklearn.cluster import kmeans_plusplus

num_tasks = 4
num_categories = 5
num_samples_per_class = 200

datas: dict[str, np.ndarray] = {}
for t in range(num_tasks):
    feature = torch.load(f'./datacache/task{t}_feature.pt', map_location='cpu', weights_only=True)
    label = torch.load(f'./datacache/task{t}_label.pt', map_location='cpu', weights_only=True)
    gen_feature = torch.load(f'./datacache/task{t}_gen_feature.pt', map_location='cpu', weights_only=True)
    untrained_gen_feature = torch.load(f'./datacache/task{t}_untrained_gen_feature.pt', map_location='cpu', weights_only=True)
    num_samples, seq_length, embeddings = feature.shape
    feature = feature.view(num_samples * seq_length, -1)
    label = label.view(num_samples * seq_length)
    gen_feature = gen_feature.view(num_samples * seq_length, -1)
    untrained_gen_feature = untrained_gen_feature.view(num_samples * seq_length, -1)
    for c in range(num_categories):
        datas[f'task{t}_class{c}_feature'] = feature[label == c].cpu().numpy()
        datas[f'task{t}_class{c}_gen_feature'] = gen_feature[label == c].cpu().numpy()
        datas[f'task{t}_class{c}_untrained_gen_feature'] = untrained_gen_feature[label == c].cpu().numpy()
for (key, value) in datas.items():
    print(key, value.shape)

task0_class0_feature (14652, 512)
task0_class0_gen_feature (14652, 512)
task0_class0_untrained_gen_feature (14652, 512)
task0_class1_feature (8799, 512)
task0_class1_gen_feature (8799, 512)
task0_class1_untrained_gen_feature (8799, 512)
task0_class2_feature (22688, 512)
task0_class2_gen_feature (22688, 512)
task0_class2_untrained_gen_feature (22688, 512)
task0_class3_feature (14261, 512)
task0_class3_gen_feature (14261, 512)
task0_class3_untrained_gen_feature (14261, 512)
task0_class4_feature (8960, 512)
task0_class4_gen_feature (8960, 512)
task0_class4_untrained_gen_feature (8960, 512)
task1_class0_feature (34159, 512)
task1_class0_gen_feature (34159, 512)
task1_class0_untrained_gen_feature (34159, 512)
task1_class1_feature (4726, 512)
task1_class1_gen_feature (4726, 512)
task1_class1_untrained_gen_feature (4726, 512)
task1_class2_feature (61397, 512)
task1_class2_gen_feature (61397, 512)
task1_class2_untrained_gen_feature (61397, 512)
task1_class3_feature (19536, 512)
task1_class3_ge

In [8]:
data_sampled: dict[str, np.ndarray] = {}
for (key, value) in tqdm(datas.items()):
    centers, _ = kmeans_plusplus(value, num_samples_per_class)
    data_sampled[key] = centers
for (key, value) in data_sampled.items():
    print(key, value.shape)

100%|██████████| 60/60 [08:43<00:00,  8.72s/it]

task0_class0_feature (200, 512)
task0_class0_gen_feature (200, 512)
task0_class0_untrained_gen_feature (200, 512)
task0_class1_feature (200, 512)
task0_class1_gen_feature (200, 512)
task0_class1_untrained_gen_feature (200, 512)
task0_class2_feature (200, 512)
task0_class2_gen_feature (200, 512)
task0_class2_untrained_gen_feature (200, 512)
task0_class3_feature (200, 512)
task0_class3_gen_feature (200, 512)
task0_class3_untrained_gen_feature (200, 512)
task0_class4_feature (200, 512)
task0_class4_gen_feature (200, 512)
task0_class4_untrained_gen_feature (200, 512)
task1_class0_feature (200, 512)
task1_class0_gen_feature (200, 512)
task1_class0_untrained_gen_feature (200, 512)
task1_class1_feature (200, 512)
task1_class1_gen_feature (200, 512)
task1_class1_untrained_gen_feature (200, 512)
task1_class2_feature (200, 512)
task1_class2_gen_feature (200, 512)
task1_class2_untrained_gen_feature (200, 512)
task1_class3_feature (200, 512)
task1_class3_gen_feature (200, 512)
task1_class3_untrain

In [10]:
np.savez('./datacache/sampled_features.npz', **data_sampled)